#Call Library

In [1]:
import pandas as pd
import numpy as np

#Import Files

In [7]:
df = pd.read_excel('/content/Silver_Data.xlsx')

In [9]:
df

,Date,Title,Account Title,Market Place,SKU,FNSKU,ASIN,Parent ASIN,Is Parent,Brand,...,Impressions,Clicks,PPC Orders,PPC Sales,PPC Cost,PPC Conv,OOE,Net Profit,Net Margin,Net ROI
0,2020-10-01,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,3423,13,2,29.36,-13.06,0.1538,0,56.28,0.1852,0.3142
1,2020-10-02,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,2786,7,0,0.00,-7.33,0.0000,0,71.73,0.2034,0.3439
2,2020-10-03,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,2927,7,1,14.69,-6.15,0.1429,0,61.37,0.1386,0.2416
3,2020-10-04,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,3584,16,2,44.07,-16.51,0.1250,0,39.49,0.1573,0.2684
4,2020-10-05,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,4329,7,0,0.00,-4.47,0.0000,0,59.86,0.2116,0.3588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13318,2021-03-27,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0,Aavalabs,...,14,0,0,0.00,0.00,0.0000,0,7.37,0.1849,0.2626
13319,2021-03-28,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0,Aavalabs,...,37,0,0,0.00,0.00,0.0000,0,13.96,0.1867,0.2493
13320,2021-03-29,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0,Aavalabs,...,16,0,0,0.00,0.00,0.0000,0,38.34,0.2800,0.4193
13321,2021-03-30,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0,Aavalabs,...,57,1,2,36.94,-0.71,2.0000,0,48.72,0.2693,0.4193


#Dim Date

In [26]:
df['Date'] = pd.to_datetime(df['Date'])
dim_date = pd.DataFrame({'Date': df['Date'].drop_duplicates().sort_values().reset_index(drop=True)})
dim_date['Date_ID'] = dim_date.index + 1
dim_date['Day'] = dim_date['Date'].dt.day
dim_date['Month'] = dim_date['Date'].dt.month
dim_date['Year'] = dim_date['Date'].dt.year
dim_date['Quarter'] = dim_date['Date'].dt.quarter
dim_date = dim_date[['Date_ID', 'Date', 'Day', 'Month',
                      'Quarter', 'Year']]

#Dim_Product

In [27]:
dim_product = df[['SKU', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand', 'Title']].drop_duplicates().reset_index(drop=True)
dim_product['SKU_ID'] = dim_product.index + 1
dim_product = dim_product[['SKU_ID', 'SKU', 'FNSKU', 'ASIN', 'Parent ASIN',
                            'Is Parent', 'Brand', 'Title']]

#Dim_Market_account

In [28]:
dim_marketplace = df[['Market Place', 'Account Title']].drop_duplicates().reset_index(drop=True)
dim_marketplace['Market_ID'] = dim_marketplace.index + 1
dim_marketplace = dim_marketplace[['Market_ID', 'Market Place','Account Title']]

#Fact

In [30]:
fact_sales = df.merge(dim_date[['Date_ID', 'Date']], on='Date', how='left')
fact_sales = fact_sales.merge(
    dim_product[['SKU_ID', 'SKU', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand', 'Title']],
    on=['SKU', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand', 'Title'],
    how='left'
)
fact_sales = fact_sales.merge(dim_marketplace, on='Market Place', how='left')
fact_sales['Sales_Key'] = fact_sales.index + 1
measure_cols = [
    'Taxes', 'Orders', 'Units', 'Refunded', 'Refund %', 'Unit Session %',
    'Promo Units', 'Organic Units', 'Per Unit Revenue', 'Revenue', 'COGS',
    'FBA Fees', 'Promo Amount', 'Sessions', 'Page Views', 'Impressions',
    'Clicks', 'PPC Orders', 'PPC Sales', 'PPC Cost', 'PPC Conv', 'OOE',
    'Net Profit', 'Net Margin', 'Net ROI'
]
fact_sales = fact_sales[
    ['Sales_Key', 'Date_ID', 'Market_ID', 'SKU_ID', 'Account Title_y'] + measure_cols
].rename(columns={'Account Title_y': 'Account Title'})

## Download Star Schema Tables

Here's the code to download the `fact_sales` table and the dimension tables (`dim_date`, `dim_product`, `dim_marketplace`) as CSV files. Each table will be saved with a descriptive filename.

In [31]:
# Save fact_sales table
fact_sales.to_csv('fact_sales.csv', index=False)
print('fact_sales.csv downloaded.')

# Save dim_date table
dim_date.to_csv('dim_date.csv', index=False)
print('dim_date.csv downloaded.')

# Save dim_product table
dim_product.to_csv('dim_product.csv', index=False)
print('dim_product.csv downloaded.')

# Save dim_marketplace table
dim_marketplace.to_csv('dim_marketplace.csv', index=False)
print('dim_marketplace.csv downloaded.')

fact_sales.csv downloaded.
dim_date.csv downloaded.
dim_product.csv downloaded.
dim_marketplace.csv downloaded.


## Combine Star Schema into a Single File (Flattened Table)

Combining a star schema into a single file effectively creates a denormalized table. While convenient for some analyses, it can lead to increased data redundancy, larger file sizes, and potential data inconsistency if not carefully managed. It negates some of the benefits of a star schema's design, such as optimized querying for specific dimensions and easier maintenance of dimension attributes.

Below is the code to merge the `fact_sales` table with its `dim_date`, `dim_product`, and `dim_marketplace` dimensions to create a single, comprehensive table.

In [23]:
# Start with a copy of the fact_sales table
flattened_df = fact_sales.copy()

# Merge with dim_date to add all date-related attributes
# 'Date_Key' is the common column. All other dim_date columns are new to flattened_df.
flattened_df = flattened_df.merge(dim_date, on='Date_Key', how='left')

# Merge with dim_product to add all product-related attributes
# 'Product_Key' is the common column. All other dim_product columns are new to flattened_df.
flattened_df = flattened_df.merge(dim_product, on='Product_Key', how='left')

# Merge with dim_marketplace to add marketplace-related attributes
# 'Marketplace_Key' is the common column.
# 'Account Title' is already in flattened_df (from fact_sales, which was renamed from 'Account Title_y').
# So, we drop 'Account Title' from dim_marketplace before merging to avoid duplicate columns (e.g., Account Title_x, Account Title_y).
flattened_df = flattened_df.merge(
    dim_marketplace.drop(columns=['Account Title']),
    on='Marketplace_Key',
    how='left'
)

# Display the first few rows of the flattened DataFrame and its shape
print("Flattened DataFrame Head:")
display(flattened_df.head())
print(f"\nFlattened DataFrame Shape: {flattened_df.shape}")

Flattened DataFrame Head:


,Sales_Key,Date_Key,Marketplace_Key,Product_Key,Account Title,Taxes,Orders,Units,Refunded,Refund %,...,Day_Name,Is_Weekend,SKU,FNSKU,ASIN,Parent ASIN,Is Parent,Brand,Title,Market Place
0,1,1,1,1,[ UK ] Aava,-50.34,17,17,0,0.0,...,Thursday,False,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,High Absorption Magnesium Citrate Supplements ...,UK
1,2,2,1,1,[ UK ] Aava,-58.89,20,20,0,0.0,...,Friday,False,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,High Absorption Magnesium Citrate Supplements ...,UK
2,3,3,1,1,[ UK ] Aava,-72.41,18,24,0,0.0,...,Saturday,True,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,High Absorption Magnesium Citrate Supplements ...,UK
3,4,4,1,1,[ UK ] Aava,-41.43,13,14,0,0.0,...,Sunday,True,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,High Absorption Magnesium Citrate Supplements ...,UK
4,5,5,1,1,[ UK ] Aava,-47.04,16,16,0,0.0,...,Monday,False,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,High Absorption Magnesium Citrate Supplements ...,UK



Flattened DataFrame Shape: (13323, 47)
